In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from scikeras.wrappers import KerasRegressor
import tensorflow as tf

from DataPreprocessing import smart_imputer, smart_jitter, df_aggregating
import Models

In [2]:
path = "/Users/zuzannasosnowska/.cache/kagglehub/datasets/krzysztofjamroz/apartment-prices-in-poland/versions/9/apartments_pl_2024_06.csv"

df = pd.read_csv(path)

df["price_per_m2"] = df["price"] / df["squareMeters"]

polish_cities_names = {'szczecin': 'Szczecin', 'gdynia': 'Gdynia', 'krakow': 'Kraków',  'poznan': 'Poznań', 'bialystok': 'Białystok',  'gdansk': 'Gdańsk', 'wroclaw': 'Wrocław', 'radom': 'Radom', 'rzeszow': 'Rzeszów', 'lodz': 'Łódź', 'katowice': 'Katowice', 'lublin': 'Lublin', 'czestochowa': 'Częstochowa', 'warszawa': 'Warszawa', 'bydgoszcz': 'Bydgoszcz'}

df['city'] = [polish_cities_names[city] for city in df['city']]

has_columns = ['hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom']
for column in has_columns:
    df[column] = df[column].eq('yes').mul(1)

df = df.drop(columns=['id'], axis=1)
df = df.drop_duplicates()


df, info = smart_imputer(df)


df = pd.get_dummies(
    df,
    columns=["type", "ownership", "buildingMaterial", "condition", "city"],
    dtype=int
)

del has_columns, column, polish_cities_names, info, path

/Users/zuzannasosnowska/PycharmProjects/UniversityProjects/Semestr 7/Praca_inżynierska/DataPreprocessing.py:105: UserWarning: Kolumna "buildingMaterial" (kategoryczna) ma >40% braków → pomijam.
  warnings.warn(f'Kolumna "{column}" (kategoryczna) ma >40% braków → pomijam.')
/Users/zuzannasosnowska/PycharmProjects/UniversityProjects/Semestr 7/Praca_inżynierska/DataPreprocessing.py:105: UserWarning: Kolumna "condition" (kategoryczna) ma >40% braków → pomijam.
  warnings.warn(f'Kolumna "{column}" (kategoryczna) ma >40% braków → pomijam.')


In [3]:
mean_match_columns = ['squareMeters', 'rooms', 'floor', 'hasParkingSpace', 'hasBalcony', 'hasStorageRoom', 'price', 'price_per_m2']

first_match_columns = df.columns.drop(mean_match_columns + ['latitude', 'longitude'])

df_agg = df_aggregating(df=df, grouped_by=['latitude', 'longitude'], aggregation={**{column: 'first' for column in first_match_columns}, **{column: 'mean' for column in mean_match_columns}})
del mean_match_columns, first_match_columns

In [4]:
y1 = df_agg["price"].values
y2 = df_agg["price_per_m2"].values
X = df_agg.drop(columns=["price", "price_per_m2", "latitude", "longitude", "geometry"], axis=1)

u = df_agg.geometry.x
v = df_agg.geometry.y
coords = list(zip(u, v))

In [5]:
def build_model(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_features,)),
        tf.keras.layers.Dense(
            30,
            activation="relu",
        ),
        tf.keras.layers.Dense(
            30,
            activation="relu",
        ),
        tf.keras.layers.Dense(
            30,
            activation="relu",
        ),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer="adam",
        loss="mse"
    )
    return model


nn_model_1 = KerasRegressor(
    model=build_model,
    model__n_features=3,
    epochs=30,
    batch_size=32,
    verbose=0
)

In [6]:
models = {
    'Linear Regression': LinearRegression(),
    'GWR bisquare': Models.SklearnGWR(kernel='bisquare'),
    'GWR gaussian': Models.SklearnGWR(kernel='gaussian'),
    'GWR exponential': Models.SklearnGWR(kernel='exponential'),
    'Random Forest': RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ),
    'Neural Network': nn_model_1,
}

In [7]:
result_df = Models.cross_validate_models(X=X[['squareMeters']], y=y1, coords=coords, models=models, n_splits=5)

In [8]:
result_df

,Linear Regression,GWR bisquare,GWR gaussian,GWR exponential,Random Forest,Neural Network
R2,0.5198,0.858731,0.847038,0.842858,0.861455,0.559664


In [9]:
print(result_df.to_latex(caption="Porównanie modeli"))

\begin{table}
\caption{Porównanie modeli}
\begin{tabular}{lrrrrrr}
\toprule
 & Linear Regression & GWR bisquare & GWR gaussian & GWR exponential & Random Forest & Neural Network \\
\midrule
R2 & 0.519800 & 0.858731 & 0.847038 & 0.842858 & 0.861455 & 0.559664 \\
\bottomrule
\end{tabular}
\end{table}



In [7]:
result_df_price_per_m2 = Models.cross_validate_models(X=X[['squareMeters']], y=y2, coords=coords, models=models, n_splits=5)

print(result_df_price_per_m2.to_latex(caption="porównanie modeli"))
result_df_price_per_m2

\begin{table}
\caption{porównanie modeli}
\begin{tabular}{lrrrrrr}
\toprule
 & Linear Regression & GWR bisquare & GWR gaussian & GWR exponential & Random Forest & Neural Network \\
\midrule
R2 & 0.115397 & 0.757705 & 0.733924 & 0.727046 & 0.787284 & 0.358456 \\
\bottomrule
\end{tabular}
\end{table}



,Linear Regression,GWR bisquare,GWR gaussian,GWR exponential,Random Forest,Neural Network
R2,0.115397,0.757705,0.733924,0.727046,0.787284,0.358456


In [8]:
result_df_price_per_m2 = Models.evaluate_model(X=X[['squareMeters']], y=y2, coords=coords, models=models)

print(result_df_price_per_m2.to_latex(caption="porównanie modeli"))
result_df_price_per_m2

\begin{table}
\caption{porównanie modeli}
\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{4}{r}{Train} & \multicolumn{4}{r}{Test} \\
 & R2 & MSE & RMSE & MAE & R2 & MSE & RMSE & MAE \\
\midrule
Linear Regression & 0.112377 & 23509104.604406 & 4848.618835 & 3797.172788 & 0.128974 & 22892991.067362 & 4784.662064 & 3739.422368 \\
GWR bisquare & 0.820548 & 4752871.042772 & 2180.108035 & 1596.124815 & 0.733155 & 7013436.516064 & 2648.289357 & 1908.968606 \\
GWR gaussian & 0.752417 & 6557348.014031 & 2560.731929 & 1921.336920 & 0.718393 & 7401407.543084 & 2720.552801 & 2033.090941 \\
GWR exponential & 0.747067 & 6699041.198006 & 2588.250606 & 1962.446642 & 0.710686 & 7603986.598985 & 2757.532701 & 2082.529260 \\
Random Forest & 0.970994 & 768248.330714 & 876.497764 & 623.556637 & 0.776634 & 5870687.988009 & 2422.950265 & 1729.804967 \\
Neural Network & 0.311022 & 18247897.360750 & 4271.755770 & 3225.255370 & 0.320738 & 17852906.001467 & 4225.269932 & 3182.014193 \\
\bottomrule
\end{tabul

Train                                              Test  \
                         R2           MSE         RMSE          MAE        R2   
Linear Regression  0.112377  2.350910e+07  4848.618835  3797.172788  0.128974   
GWR bisquare       0.820548  4.752871e+06  2180.108035  1596.124815  0.733155   
GWR gaussian       0.752417  6.557348e+06  2560.731929  1921.336920  0.718393   
GWR exponential    0.747067  6.699041e+06  2588.250606  1962.446642  0.710686   
Random Forest      0.970994  7.682483e+05   876.497764   623.556637  0.776634   
Neural Network     0.311022  1.824790e+07  4271.755770  3225.255370  0.320738   

                                                           
                            MSE         RMSE          MAE  
Linear Regression  2.289299e+07  4784.662064  3739.422368  
GWR bisquare       7.013437e+06  2648.289357  1908.968606  
GWR gaussian       7.401408e+06  2720.552801  2033.090941  
GWR exponential    7.603987e+06  2757.532701  2082.529260  
Random Forest      5.870688e+06  2422.950265  1729.804967  
Neural Network     1.785291e+07  4225.269932  3182.014193

In [14]:
print(result_df_price_per_m2.to_latex(caption="porównanie modeli", float_format="%.2f"))

\begin{table}
\caption{porównanie modeli}
\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{4}{r}{Train} & \multicolumn{4}{r}{Test} \\
 & R2 & MSE & RMSE & MAE & R2 & MSE & RMSE & MAE \\
\midrule
Linear Regression & 0.11 & 23509104.60 & 4848.62 & 3797.17 & 0.13 & 22892991.07 & 4784.66 & 3739.42 \\
GWR bisquare & 0.82 & 4752871.04 & 2180.11 & 1596.12 & 0.73 & 7013436.52 & 2648.29 & 1908.97 \\
GWR gaussian & 0.75 & 6557348.01 & 2560.73 & 1921.34 & 0.72 & 7401407.54 & 2720.55 & 2033.09 \\
GWR exponential & 0.75 & 6699041.20 & 2588.25 & 1962.45 & 0.71 & 7603986.60 & 2757.53 & 2082.53 \\
Random Forest & 0.97 & 768248.33 & 876.50 & 623.56 & 0.78 & 5870687.99 & 2422.95 & 1729.80 \\
Neural Network & 0.31 & 18247897.36 & 4271.76 & 3225.26 & 0.32 & 17852906.00 & 4225.27 & 3182.01 \\
\bottomrule
\end{tabular}
\end{table}



In [10]:
result_df = Models.evaluate_model(X=X[['squareMeters']], y=y1, coords=coords, models=models)

In [11]:
result_df

Train                                              \
                         R2           MSE           RMSE            MAE   
Linear Regression  0.520827  9.080834e+10  301344.219188  221150.484720   
GWR bisquare       0.902542  1.846940e+10  135902.160040   92306.751823   
GWR gaussian       0.861856  2.617979e+10  161801.702535  111572.487710   
GWR exponential    0.859009  2.671919e+10  163460.057929  114012.343883   
Random Forest      0.981377  3.529303e+09   59407.940246   38627.999289   
Neural Network     0.556261  8.409310e+10  289988.101629  205170.803544   

                       Test                                              
                         R2           MSE           RMSE            MAE  
Linear Regression  0.518730  9.203730e+10  303376.496785  221477.015314  
GWR bisquare       0.843750  2.988096e+10  172861.108542  112583.328334  
GWR gaussian       0.833968  3.175167e+10  178189.992372  120484.495586  
GWR exponential    0.828505  3.279645e+10  181097.906372  123656.263918  
Random Forest      0.852418  2.822331e+10  167997.942277  109800.201950  
Neural Network     0.554096  8.527379e+10  292016.756364  206516.006825

In [12]:
print(result_df.to_latex(caption="Porównanie modeli"))

\begin{table}
\caption{Porównanie modeli}
\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{4}{r}{Train} & \multicolumn{4}{r}{Test} \\
 & R2 & MSE & RMSE & MAE & R2 & MSE & RMSE & MAE \\
\midrule
Linear Regression & 0.520827 & 90808338437.736618 & 301344.219188 & 221150.484720 & 0.518730 & 92037298801.436371 & 303376.496785 & 221477.015314 \\
GWR bisquare & 0.902542 & 18469397103.456062 & 135902.160040 & 92306.751823 & 0.843750 & 29880962846.350624 & 172861.108542 & 112583.328334 \\
GWR gaussian & 0.861856 & 26179790943.144226 & 161801.702535 & 111572.487710 & 0.833968 & 31751673381.477077 & 178189.992372 & 120484.495586 \\
GWR exponential & 0.859009 & 26719190538.158501 & 163460.057929 & 114012.343883 & 0.828505 & 32796451692.453991 & 181097.906372 & 123656.263918 \\
Random Forest & 0.981377 & 3529303364.308768 & 59407.940246 & 38627.999289 & 0.852418 & 28223308609.298710 & 167997.942277 & 109800.201950 \\
Neural Network & 0.556261 & 84093099086.174408 & 289988.101629 & 205170.80354

In [16]:
nn_model_2 = KerasRegressor(
    model=build_model,
    model__n_features=X.shape[1]+2,
    epochs=30,
    batch_size=32,
    verbose=0,
)

models = {
    'Linear Regression': LinearRegression(),
    # 'Ridge Regression': Ridge(alpha=1.0),
    # 'Lasso Regression': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ),
    'Neural Network': nn_model_2,
}

In [ ]:
result2_df = Models.evaluate_model(X=X.values, y=y1, coords=coords, models=models)

print(result2_df.to_latex(caption="Porównanie modeli"))

result2_df

In [14]:
print(result2_df.to_latex(caption="Porównanie modeli", formatters={'MSE': lambda x: f"{x:.2f}", "RMSE": lambda x: f"{x:.2f}", "MAE": lambda x: f"{x:.2f}"}))

\begin{table}
\caption{Porównanie modeli}
\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{4}{r}{Train} & \multicolumn{4}{r}{Test} \\
 & R2 & MSE & RMSE & MAE & R2 & MSE & RMSE & MAE \\
\midrule
Linear Regression & 0.795238 & 38804484871.666000 & 196988.539950 & 137427.515940 & 0.791045 & 39960266316.310783 & 199900.641110 & 140810.763050 \\
Random Forest & 0.985530 & 2742121492.797196 & 52365.269910 & 33263.750485 & 0.893215 & 20421321927.907066 & 142903.190755 & 92189.491581 \\
Neural Network & 0.862306 & 26094523380.930225 & 161537.993614 & 109394.010947 & 0.850872 & 28518892100.424446 & 168875.374464 & 114661.976259 \\
\bottomrule
\end{tabular}
\end{table}



In [17]:
result2_df = Models.evaluate_model(X=X.values, y=y2, coords=coords, models=models)

print(result2_df.to_latex(caption="Porównanie modeli"))

result2_df

\begin{table}
\caption{Porównanie modeli}
\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{4}{r}{Train} & \multicolumn{4}{r}{Test} \\
 & R2 & MSE & RMSE & MAE & R2 & MSE & RMSE & MAE \\
\midrule
Linear Regression & 0.737808 & 6944269.086880 & 2635.198112 & 1980.211031 & 0.734161 & 6986983.609485 & 2643.290300 & 2002.925121 \\
Random Forest & 0.979420 & 545062.210342 & 738.283286 & 525.163541 & 0.844419 & 4089112.884188 & 2022.155504 & 1457.168662 \\
Neural Network & 0.804891 & 5167566.130286 & 2273.228130 & 1670.468517 & 0.785882 & 5627609.546012 & 2372.258322 & 1766.175433 \\
\bottomrule
\end{tabular}
\end{table}



Train                                              Test  \
                         R2           MSE         RMSE          MAE        R2   
Linear Regression  0.737808  6.944269e+06  2635.198112  1980.211031  0.734161   
Random Forest      0.979420  5.450622e+05   738.283286   525.163541  0.844419   
Neural Network     0.804891  5.167566e+06  2273.228130  1670.468517  0.785882   

                                                           
                            MSE         RMSE          MAE  
Linear Regression  6.986984e+06  2643.290300  2002.925121  
Random Forest      4.089113e+06  2022.155504  1457.168662  
Neural Network     5.627610e+06  2372.258322  1766.175433

In [18]:
print(result2_df.to_latex(caption="porównanie modeli", float_format="%.2f"))

\begin{table}
\caption{porównanie modeli}
\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{4}{r}{Train} & \multicolumn{4}{r}{Test} \\
 & R2 & MSE & RMSE & MAE & R2 & MSE & RMSE & MAE \\
\midrule
Linear Regression & 0.74 & 6944269.09 & 2635.20 & 1980.21 & 0.73 & 6986983.61 & 2643.29 & 2002.93 \\
Random Forest & 0.98 & 545062.21 & 738.28 & 525.16 & 0.84 & 4089112.88 & 2022.16 & 1457.17 \\
Neural Network & 0.80 & 5167566.13 & 2273.23 & 1670.47 & 0.79 & 5627609.55 & 2372.26 & 1766.18 \\
\bottomrule
\end{tabular}
\end{table}

